In [7]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, date
import smtplib 
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from email import encoders  
from email.message import EmailMessage
import os
from dotenv import load_dotenv

In [95]:
login_list = """ 'v.popkova', 'l.troitskaya', 't.badikova',
                'n.reutova', 'a.kokorev','ea.kuznetsova',
                'a.yakubenko', 'a.zaharova', 'a.shichalina',
                'o.safronov', 'a.fomchenkov', 'm.toporkova',
                'd.kondratenko', 'o.mezenceva', 'k.arakelyan',
                'p.korenitsyn', 'a.shmelev', 'e.stepina','p.kartashkov',
                'n.radchenko', 'e.sokolova', 'd.petuhov' ,
                'a.smogunov', 's.kovalchuk' """

    # condition of export
today_condition = " ca.start_date::date <= current_date and ca.end_date::date >= current_date "
today_condition_bad = " ba.start_dt::date <= current_date and ba.end_dt::date >= current_date "

next_week_condition = """
        extract(week from ca.start_date) = extract(week from current_date)+1
            or extract(week from ca.end_date) = extract(week from current_date)+1
            or (extract(week from ca.start_date)<extract(week from current_date)+1
                and extract(week from ca.end_date) > extract(week from current_date)+1) """
next_week_condition_bad = """
        extract(week from ba.start_dt) = extract(week from current_date)+1
            or extract(week from ba.end_dt) = extract(week from current_date)+1
            or (extract(week from ba.start_dt)<extract(week from current_date)+1
                and extract(week from ba.end_dt) > extract(week from current_date)+1) """

def bad_sql ():
    return f"""
        select bat.info as abc_type, 
        	e.name as empl,
        	ba.start_dt as st_dt,
        	ba.end_dt as end_dt,
            ba.stage
        from dds.employees e
            join dds.departments d on e.dept_id = d.dept_id
            join data_store.b_absence ba on ba.absent_employee = e.name and str_lvl = '1'
            and extract(year from ba.start_dt) = extract(year from current_date)
            and (extract(week from ba.start_dt) = extract(week from current_date)+1
                    or extract(week from ba.end_dt) = extract(week from current_date)+1
                    or (extract(week from ba.start_dt)<extract(week from current_date)+1
                        and extract(week from ba.end_dt) > extract(week from current_date)+1)
            or (ba.start_dt::date <= current_date and ba.end_dt::date >= current_date))
            left join data_store.b_absence_type bat on bat.typeofabsence = ba.type_of_absence 
        order by st_dt
    """

def gimme_sql (condition):
        return f"""
     select
        case when lower(ca.typeofabsence) like '%%отпуск%%' then 'Отпуск'
            when lower(ca.typeofabsence) like '%%больничный%%' then 'Больничный'
            when lower(ca.typeofabsence) like '%%командир%%' then 'Командировка'
            when lower(ca.typeofabsence) like '%%дополнительные выходные дни%%' then 'Дополнительные выходные'
            when lower(ca.typeofabsence) like '%%болезнь%%' then 'Больничный'
            when lower(ca.typeofabsence) like '%%тсутствие с сохранением оплаты%%' then 'Отсутствие'
            when lower(ca.typeofabsence) like '%%тсутствие по невыясненным причинам%%' then 'Отсутствие'
            else bat.info end as abc_type,
        e.name as empl,
        case
            when length(ca.typeofabsence)<=4 and ca.start_date::time ='00:00:00'
                then to_char(ca.start_date::date +'09:00:00'::time,'dd.mm.yyyy HH24:MI')
            when length(ca.typeofabsence)<=4 then to_char(ca.start_date,'dd.mm.yyyy HH24:MI')
            else to_char(ca.start_date::date,'dd.mm.yyyy') end as st_dt,
        case
            when length(ca.typeofabsence)<=4 and ca.end_date::time ='00:00:00'
                then to_char(ca.end_date::date +'18:00:00'::time,'dd.mm.yyyy HH24:MI')
            when length(ca.typeofabsence)<=4 then to_char(ca.end_date,'dd.mm.yyyy HH24:MI')
            else to_char(ca.end_date::date,'dd.mm.yyyy') end as end_dt
    from dds.employees e
    join dds.departments d on e.dept_id = d.dept_id
    join data_store.c_employees ce on ce.id = e.tabel_number
    join data_store.c_absences ca on ca.cardnumber = e.tabel_number
    left join data_store.b_absence_type bat on bat.typeofabsence = ca.typeofabsence
    where ce.deleted = False
        and ce.id <> '0000000722'
        and (e.str_lvl in (1, 2, 3) --and e.no_stat=False
                and e.login not in ('o.doshechkina', 'n.trubnikova', 'a.pershikova') or e.login in ({login_list}))
        and ({condition})
        and extract(year from start_date) = extract(year from current_date)
        and ca.start_date <= ca.end_date
    order by 2, 1, start_date, end_date;
    """

sql_heads_email = f"""
    SELECT
        distinct split_part(e.name, ' ', 2) || ' ' || split_part(e.name, ' ', 3) as name,
        be.email
    from dds.employees e
    join data_store.b_employees be on be."name"  =e."name"
    join data_store.c_employees ce on e.uniq_id = ce.uniq_id and ce.deleted = False
    where (e.str_lvl = 1 or e.login  in ({login_list+", 'a.pershikova', 'o.doshechkina', 'm.gromova' "}) )
        and e.uniq_id  not in
        ( --minus people on vocation
            select distinct e.uniq_id
            from dds.employees e
            join data_store.c_absences ca on ca.cardnumber = e.tabel_number
            where
                (ca.start_date <= current_date and ca.end_date >= current_date  )
                and extract(year from start_date) = extract(year from current_date)
                and ca.start_date <= ca.end_date
                and e.login <> 'a.zaharova'
            )
         or e.login = 'a.pershikova';"""

class Mailer:
    def __init__(self):
        self.__db_env_names = ["DB_HOST", "DB_PORT", "DB_USER", "DB_PASSWORD", "DB_NAME"]
        self.__mail_env_names = ["ADDR_FROM", "MAIL_PASSWORD"]
        self.__months = ['январь', 'февраль', 'март', 'апрель', 'апрель', 'май', 'июнь', 'июль', 
                         'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь']
        self.__set_engine()

    def __get_env(self, v_names):
        load_dotenv()
        return [os.getenv(x) for x in v_names]
        
    def __set_engine(self):
        host, port, user, password, db_name = self.__get_env(self.__db_env_names)
        self.__engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_name}", pool_pre_ping=True)

    def __get_query(self, query):
        with self.__engine.connect() as con:
            df = pd.read_sql_query(query, con)
        return df

    def test(self):
        self.__today_abc = self.__get_query(gimme_sql(today_condition))
        self.__today_abc.columns = ['Вид отсутствия', 'ФИО работника','Дата начала','Дата окончания']

        self.__next_abc = self.__get_query(gimme_sql(next_week_condition))
        self.__next_abc.columns = ['Вид отсутствия', 'ФИО работника','Дата начала','Дата окончания']
        
        self.__heads = self.__get_query(sql_heads_email)
        self.__heads.columns = ['name','email']

        self.__heads_absence = self.__get_query(bad_sql())
        self.__heads_absence.columns = ['Вид отсутствия', 'ФИО работника','Дата начала','Дата окончания', 'Стадия']
        return self.__today_abc, self.__next_abc, self.__heads, self.__heads_absence


mailer = Mailer()

In [96]:
mailer.test()

(             Вид отсутствия                       ФИО работника  \
 0                    Отпуск          Аликина Наталья Алексеевна   
 1                    Отпуск             Бенгер Арсений Эмилевич   
 2          Удаленная работа         Вельможин Сергей Николаевич   
 3                    Отпуск    Виноградов Всеволод Владимирович   
 4                    Другое      Воропаев Дмитрий Александрович   
 5                    Отпуск            Горелова Елена Сергеевна   
 6                    Отпуск        Громова Марина Святославовна   
 7                    Отпуск          Грудцов Иван Александрович   
 8                Больничный  Демьянкина Анастасия Александровна   
 9              Командировка            Иванов Алексей Сергеевич   
 10             Командировка           Иванов Дмитрий Викторович   
 11                   Отпуск          Ивашина Наталья Михайловна   
 12         Удаленная работа             Ильина Наталья Павловна   
 13                   Другое     Казорина Анаста

In [97]:



# Global constants
postgres_conn_id = 'Skid'
day = datetime.today().date()
start = day - timedelta(days=day.weekday())+timedelta(days=7)
end = start + timedelta(days=6)

addr_from = "no-reply@rt-techpriemka.ru"
password  = "whtzFzZ5gpvTjvDYetiy"


def att_mes_send (df, addr_to, one_day, heads, next_df):
    '''Creating letter if absences exist'''
    # create text
    msg_text = f"""
    <html>
        <p>Добрый день, {'dfdf'}!<br /><br />
            Ежедневная рассылка об отсутствии работников за {one_day.strftime("%d.%m.%Y")}г. (для учета в работе).
            <br/>
            {df.to_html(index = False, justify = 'center')}
            <br/>
            {'' if next_df is None else f"Запланированные отсутствия <b>на следующей неделе</b>.<br /><br/>" +
                f"Отсутствия в период с {start} по {end} включительно:"+
                f"{next_df.to_html(index = False, justify = 'center')}<br />" }
            <br/>
            Оперативную информацию о фактических отсутствиях руководителей в офисе Вы можете <a href="https://skid.rtt.digital:8443/public/dashboard/8fabe3cb-41d7-4a26-9934-3d5069e42094"> <i>увидеть на дэшборде  на вкладке "Руководители"</i></a>. <br/>
На вкладке "Работники" отражены текущие отсутсвия работников и запланированные на ближайший месяц</br>
            <i>(Дэшборд доступен из внутренней сети Общества. В предупреждении от mail.ru нажимте Подробнее, затем перейти по ссылке)</i> <br/><br/>
            Желаем {pd.Series(['хорошего', 'продуктивного', 'отличного',
                    'плодотворного', 'удачного', 'успешного']).sample(1).values[0]} дня!<br /><br />
            </div>
            <sub>Это ежедневная рассылка. Вы всегда можете отписаться от нее, просто позвонив по тел. 269 или написав письмо на почту d.kondratenko@rt-techpriemka.ru.</sub></p>
            <sub>Пожалуйста, не отвечайте на это письмо.</sub></p>
            </html>"""

    msg_subj = f"""Информация об отсутствиях {'за сегодня' if next_df is None else 'на следующей неделе и сегодня'}"""
    # Compose message
    msg = MIMEMultipart()
    msg['From'] = addr_from
    msg['To'] = addr_to
    msg['Subject'] = msg_subj
    msg.attach(MIMEText(msg_text, 'html'))

    # Send mail
    server = smtplib.SMTP_SSL('217.69.139.160', 465)
    server.login(addr_from, password)
    server.sendmail(addr_from , addr_to, msg.as_string())
    server.quit()

def noatt_mes_send (addr_to, heads, next_df):
    '''Creating info-letter'''
    # create text
    msg_text = f"""
    <html>
        <p>Добрый день, {'dfdf'}!<br /><br />
            <br/>
            Сегодня все работники на своих рабочих местах!
            <br><br/>
            {'' if next_df is None else f"Запланированные отсутствия <b>на следующей неделе</b>.<br /><br/>" +
                f"Отсутствия в период с {start} по {end} включительно:" +
                f"{next_df.to_html(index = False, justify = 'center')} <br />" }
            Оперативную информацию о фактических отсутствиях руководителей в офисе Вы можете <a href="https://skid.rtt.digital:8443/public/dashboard/8fabe3cb-41d7-4a26-9934-3d5069e42094"> <i>увидеть на дэшборде</i> </a>на вкладке "Руководители". <br/>
            На вкладке "Работники" отражены текущие отсутствия и запланированные на ближайший месяц.</br>
            <i>(Дэшборд доступен из внутренней сети Общества. В предупреждении от mail.ru нажимте Подробнее, затем перейти по ссылке)</i> <br/><br/>
            Желаем успехов и хорошего дня!
            <br /><br />
            </div>
            <sub>Это ежедневная рассылка. Вы всегда можете отписаться от нее, просто позвонив по тел. 269 или написав письмо на почту d.kondratenko@rt-techpriemka.ru.</sub></p>
            <sub>Это информационное письмо, пожалуйста, не отвечайте на него.</sub></p>
        </html>"""

    # Compose message
    msg_subj = f"""Информация об отсутствиях {'за сегодня' if next_df is None else 'на следующей неделе и сегодня'}"""
    msg = MIMEMultipart()
    msg['From'] = addr_from
    msg['To'] = addr_to
    msg['Subject'] = msg_subj
    msg.attach(MIMEText(msg_text, 'html'))

    # Send mail
    server = smtplib.SMTP_SSL('217.69.139.160', 465)
    server.login(addr_from, password)
    server.sendmail(addr_from , addr_to, msg.as_string())
    server.quit()


def send_to_sec(df, one_day):
    '''Creating letter if absences exist'''
    # create text
    msg_text = f"""
    <html>
        <p>Добрый день, Татьяна Сергеевна!<br /><br />
            Ежедневная рассылка об отсутствии ЗГД и ГД АО «Техпоставка» за {one_day.strftime("%d.%m.%Y")}г. + следующая неделя (для учета в работе).
            <br/>
            {df.to_html(index = False, justify = 'center')}
            <br/>
            
            Оперативную информацию о фактических отсутствиях руководителей в офисе Вы можете <a href="https://skid.rtt.digital:8443/public/dashboard/8fabe3cb-41d7-4a26-9934-3d5069e42094"> <i>увидеть на дэшборде  на вкладке "Руководители"</i></a>. <br/>
    На вкладке "Работники" отражены текущие отсутсвия работников и запланированные на ближайший месяц</br>
            <i>(Дэшборд доступен из внутренней сети Общества. В предупреждении от mail.ru нажимте Подробнее, затем перейти по ссылке)</i> <br/><br/>
            Желаем {pd.Series(['хорошего', 'продуктивного', 'отличного',
                    'плодотворного', 'удачного', 'успешного']).sample(1).values[0]} дня!<br /><br />
            </div>
            <sub>Это ежедневная рассылка. Вы всегда можете отписаться от нее, просто позвонив по тел. 269 или написав письмо на почту d.kondratenko@rt-techpriemka.ru.</sub></p>
            <sub>Пожалуйста, не отвечайте на это письмо.</sub></p>
            </html>"""

    addr_to = 'd.konshin@rt-techpriemka.ru'

    msg_subj = f"""Информация об отсутствиях {'за сегодня' if df is None else 'на следующей неделе и сегодня'}"""
    # Compose message
    msg = MIMEMultipart()
    msg['From'] = addr_from
    # msg['To'] = 't.badikova@rt-techpriemka.ru'
    msg['To'] = addr_to
    msg['Subject'] = msg_subj
    msg.attach(MIMEText(msg_text, 'html'))

    # Send mail
    server = smtplib.SMTP_SSL('217.69.139.160', 465)
    server.login(addr_from, password)
    server.sendmail(addr_from , addr_to, msg.as_string())
    server.quit()


def massive_attack():
    conn = 'Skid'
    day = datetime.today().date()
    '''Function for sending a messages'''
    # today_abc, next_abc, heads = read_data_sql(conn)
    today_abc, next_abc, heads, bad_df = mailer.test()
    heads = pd.concat([heads, pd.DataFrame(data = {'name': ['Александра Сергеевна'],
                                       'email': ['a.shichalina@rt-globalconsult.ru']})])\
                    .reset_index(drop=True)

    ulist = ['d.konshin@rt-techpriemka.ru']
#    ulist = ['d.kondratenko@rt-techpriemka.ru']
    # ulist = heads['email'].unique()
    print('Dataframes were loaded')

    # sending letters for all days except Friday
    if datetime.today().weekday() !=4:
        # if there is not Friday and absences exist
        if bad_df.shape[0] > 0:
            send_to_sec(bad_df, day)
        if len(today_abc) > 0:
            for i in range(len(ulist)):
                send_msg = 'Letters were sent to '
                try:
                    att_mes_send (today_abc, ulist[i], day, heads, None)
                    send_msg += str(ulist[i]) + ' '
                except:
                    print('Сообщение не отправлено по адресу: ', ulist[i])
                print(send_msg)

        # if there is not Friday and absences doesn't exist
        else:
            send_msg = 'Info letters were sent to '
            for i in range(len(ulist)):
                try:
                    noatt_mes_send( ulist[i], heads, None)
                    send_msg += str(ulist[i]) + ' '
                except:
                    print('Сообщение не отправлено по адресу: ', ulist[i])
                print(send_msg)

    # sending letters on Friday (with next week absences)
    else:
        if bad_df.shape[0] > 0:
            send_to_sec(bad_df, day)
        # if there is Friday and absences exist
        if len(today_abc) > 0:
            send_msg = 'Letters were sent to '
            for i in range(len(ulist)):
                # try:
                att_mes_send (today_abc, ulist[i], day, heads, next_abc)
                send_msg += str(ulist[i]) + ' '
                # except:
                    # print('Сообщение не отправлено по адресу: ', ulist[i])
                print(send_msg)

        # if there is Friday and absences doesn't exist
        else:
            send_msg = 'Info letters were sent to '
            for i in range(len(ulist)):
                try:
                    noatt_mes_send( ulist[i], heads, next_abc)
                    send_msg += str(ulist[i]) + ' '
                except:
                    print('Сообщение не отправлено по адресу: ', ulist[i])
                print(send_msg)

    print('###### END of script with head adsenses', datetime.now(), '######')

In [98]:
massive_attack()

Dataframes were loaded
Letters were sent to d.konshin@rt-techpriemka.ru 
###### END of script with head adsenses 2026-03-20 15:49:47.556250 ######
